# Чистовик: CatBoost, квантили 0.1 / 0.5 / 0.9

Читает `data/`, считает CV по пространственным блокам внутри города (точки теста новые), пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой.

In [ ]:
import json, sys
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
TARGET = "daily_orders"
y = tr[TARGET].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)


def make_features(df):
    X = df.drop(columns=[c for c in ("id", TARGET) if c in df.columns]).copy()
    X["floor"] = X["floor"].astype(int)
    return X


Xtr = make_features(tr)
Xte = make_features(te)
CATS = ["city", "district_type", "entrance_type"]


def predict(Xa, ya, Xb):
    """Три квантильные модели CatBoost на log1p(y), затем монотонность и клип."""
    out = []
    for a in QUANTILES:
        m = CatBoostRegressor(
            loss_function=f"Quantile:alpha={a}",
            iterations=1000,
            learning_rate=0.03,
            depth=4,
            l2_leaf_reg=20,
            random_seed=0,
            thread_count=2,
            verbose=0,
            allow_writing_files=False,
            cat_features=CATS,
        )
        m.fit(Xa, np.log1p(ya))
        out.append(np.expm1(m.predict(Xb)))
    q = np.sort(np.column_stack(out), axis=1)
    return np.clip(q, 0, 5000)


# CV: точки теста новые, соседние точки похожи -> фолды по пространственным блокам внутри города.
# Блоки — сетка 3x3 по квантилям lat/lon города, без KMeans: разбиение не зависит от версии sklearn.
fold_id = np.zeros(len(tr), dtype=int)
for k, (city, idx) in enumerate(sorted(tr.groupby("city").indices.items())):
    d = tr.iloc[idx]
    block = pd.qcut(d["latitude"], 3, labels=False, duplicates="drop").to_numpy() * 3 \
        + pd.qcut(d["longitude"], 3, labels=False, duplicates="drop").to_numpy()
    fold_id[idx] = (block + k) % 4
splits = [(np.where(fold_id != f)[0], np.where(fold_id == f)[0]) for f in range(4)]

folds = []
for a, b in splits:
    p = predict(Xtr.iloc[a], y[a], Xtr.iloc[b])
    folds.append(100 * quantile_quality(y[b], p[:, 0], p[:, 1], p[:, 2], tr["city"].iloc[b]))
print("folds:", [round(f, 3) for f in folds])

# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test = np.round(predict(Xtr, y, Xte), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
assert np.isfinite(sub.iloc[:, 1:].to_numpy()).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("catboost_spatial", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                          # последняя строка — для проверки воспроизводимости
